# Adaptive Model Routing Serving Optimizer - Exploration

This notebook demonstrates the key components of the adaptive model routing system and provides interactive exploration of the routing algorithms.

In [ ]:
import sys
from pathlib import Path

# Add src to path
sys.path.append(str(Path("..") / "src"))

import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from adaptive_model_routing_serving_optimizer.utils.config import load_config
from adaptive_model_routing_serving_optimizer.models.model import AdaptiveRoutingModel
from adaptive_model_routing_serving_optimizer.data.loader import SyntheticDataLoader
from adaptive_model_routing_serving_optimizer.evaluation.metrics import RoutingMetrics

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

## 1. Configuration and Model Setup

In [ ]:
# Load configuration
config = load_config("../configs/default.yaml")
print("Configuration loaded successfully")
print(f"Model architectures: {config['model']['architectures']}")
print(f"Compression variants: {[v['type'] for v in config['model']['compression_variants']]}")

# Initialize model
model = AdaptiveRoutingModel(config._config)
print(f"\nModel initialized with {model.num_variants} variants")
print(f"Context dimension: {model.context_dim}")

## 2. Explore Model Variants

In [ ]:
# Explore compression variants and their characteristics
variant_info = []
for i in range(model.num_variants):
    info = model.variant_manager.get_variant_info(i)
    variant_info.append(info)
    print(f"Variant {i}: {info['name']}")
    print(f"  Memory multiplier: {info['memory_multiplier']}")
    print(f"  Latency multiplier: {info['latency_multiplier']}")
    print(f"  Precision: {info['precision']}")
    print()

In [ ]:
# Visualize variant trade-offs
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

variants = [info['name'] for info in variant_info]
memory_multipliers = [info['memory_multiplier'] for info in variant_info]
latency_multipliers = [info['latency_multiplier'] for info in variant_info]

# Memory vs Latency trade-off
axes[0].scatter(memory_multipliers, latency_multipliers, s=100)
for i, variant in enumerate(variants):
    axes[0].annotate(variant, (memory_multipliers[i], latency_multipliers[i]), 
                    xytext=(5, 5), textcoords='offset points')
axes[0].set_xlabel('Memory Multiplier')
axes[0].set_ylabel('Latency Multiplier')
axes[0].set_title('Memory vs Latency Trade-off')

# Bar chart of characteristics
x = np.arange(len(variants))
width = 0.35
axes[1].bar(x - width/2, memory_multipliers, width, label='Memory', alpha=0.8)
axes[1].bar(x + width/2, latency_multipliers, width, label='Latency', alpha=0.8)
axes[1].set_xlabel('Variants')
axes[1].set_ylabel('Multiplier')
axes[1].set_title('Variant Characteristics')
axes[1].set_xticks(x)
axes[1].set_xticklabels(variants, rotation=45)
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. Contextual Bandit Exploration

In [ ]:
# Test different exploration strategies
exploration_algorithms = ['epsilon_greedy', 'ucb', 'thompson_sampling']
num_steps = 500

results = {}

for algorithm in exploration_algorithms:
    print(f"Testing {algorithm} algorithm")
    
    # Create temporary config with this algorithm
    temp_config = config._config.copy()
    temp_config['training']['bandit']['algorithm'] = algorithm
    temp_model = AdaptiveRoutingModel(temp_config)
    
    decisions = []
    rewards = []
    regrets = []
    
    for step in range(num_steps):
        # Generate random context
        context = torch.randn(model.context_dim)
        
        # Get bandit decision
        variant = temp_model.select_model_variant(context, use_bandit=True)
        decisions.append(variant)
        
        # Simulate performance and reward
        request_data = {'batch_size': 1, 'image_size': 224, 'priority': 3}
        performance = temp_model.variant_manager.estimate_performance(variant, request_data)
        
        # Add noise
        performance['latency_ms'] += np.random.normal(0, 5)
        performance['accuracy'] += np.random.normal(0, 0.005)
        
        sla_constraints = temp_config['routing']['sla_constraints']
        reward = temp_model.variant_manager.calculate_reward(variant, performance, sla_constraints)
        rewards.append(reward)
        
        # Calculate regret (difference from oracle)
        oracle_variant = temp_model.variant_manager.get_best_variant_for_context(request_data, sla_constraints)
        oracle_performance = temp_model.variant_manager.estimate_performance(oracle_variant, request_data)
        oracle_reward = temp_model.variant_manager.calculate_reward(oracle_variant, oracle_performance, sla_constraints)
        
        regret = max(0, oracle_reward - reward)
        regrets.append(regret)
        
        # Update bandit
        temp_model.update_with_feedback(context, variant, performance)
    
    results[algorithm] = {
        'decisions': decisions,
        'rewards': rewards,
        'regrets': regrets,
        'cumulative_regret': np.cumsum(regrets)
    }

print("Exploration analysis complete")

In [ ]:
# Visualize exploration results
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Cumulative regret
for algorithm in exploration_algorithms:
    axes[0, 0].plot(results[algorithm]['cumulative_regret'], label=algorithm)
axes[0, 0].set_xlabel('Steps')
axes[0, 0].set_ylabel('Cumulative Regret')
axes[0, 0].set_title('Learning Efficiency (Lower is Better)')
axes[0, 0].legend()

# Average reward over time
window_size = 50
for algorithm in exploration_algorithms:
    rewards = results[algorithm]['rewards']
    smoothed = np.convolve(rewards, np.ones(window_size)/window_size, mode='valid')
    axes[0, 1].plot(smoothed, label=algorithm)
axes[0, 1].set_xlabel('Steps')
axes[0, 1].set_ylabel('Average Reward')
axes[0, 1].set_title(f'Reward Over Time (Moving Average, window={window_size})')
axes[0, 1].legend()

# Decision distribution
for i, algorithm in enumerate(exploration_algorithms):
    decisions = results[algorithm]['decisions']
    unique, counts = np.unique(decisions, return_counts=True)
    
    ax = axes[1, 0] if i == 0 else axes[1, 1] if i == 1 else None
    if ax and i < 2:  # Only plot first two algorithms
        ax.bar(unique, counts)
        ax.set_xlabel('Variant ID')
        ax.set_ylabel('Count')
        ax.set_title(f'Decision Distribution - {algorithm}')

# Final algorithm comparison (use remaining subplot)
if len(exploration_algorithms) >= 3:
    decisions = results[exploration_algorithms[2]]['decisions']
    unique, counts = np.unique(decisions, return_counts=True)
    axes[1, 1].bar(unique, counts)
    axes[1, 1].set_xlabel('Variant ID')
    axes[1, 1].set_ylabel('Count')
    axes[1, 1].set_title(f'Decision Distribution - {exploration_algorithms[2]}')

plt.tight_layout()
plt.show()

## 4. Context Analysis

In [ ]:
# Generate diverse contexts and analyze routing patterns
data_loader = SyntheticDataLoader(config._config)
num_contexts = 1000

contexts = []
policy_decisions = []
bandit_decisions = []

for _ in range(num_contexts):
    # Generate context
    context_dict = data_loader.generate_request_context(batch_size=1)
    
    # Convert to tensor (simplified)
    context_tensor = torch.randn(model.context_dim)  # Placeholder for actual context extraction
    contexts.append(context_tensor)
    
    # Get routing decisions
    policy_variant = model.select_model_variant(context_tensor, use_bandit=False)
    bandit_variant = model.select_model_variant(context_tensor, use_bandit=True)
    
    policy_decisions.append(policy_variant)
    bandit_decisions.append(bandit_variant)

print(f"Generated {num_contexts} context samples")

In [ ]:
# Analyze decision patterns
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Policy vs Bandit decision agreement
agreement = [p == b for p, b in zip(policy_decisions, bandit_decisions)]
agreement_rate = np.mean(agreement)

axes[0, 0].hist([0, 1], bins=2, weights=[1-agreement_rate, agreement_rate], 
               alpha=0.7, color=['red', 'green'])
axes[0, 0].set_xticks([0, 1])
axes[0, 0].set_xticklabels(['Disagree', 'Agree'])
axes[0, 0].set_ylabel('Fraction')
axes[0, 0].set_title(f'Policy vs Bandit Agreement ({agreement_rate:.2%})')

# Decision distributions
unique_policy, counts_policy = np.unique(policy_decisions, return_counts=True)
unique_bandit, counts_bandit = np.unique(bandit_decisions, return_counts=True)

x = np.arange(model.num_variants)
width = 0.35

# Ensure all variants are represented
policy_counts = np.zeros(model.num_variants)
bandit_counts = np.zeros(model.num_variants)

for i, count in zip(unique_policy, counts_policy):
    policy_counts[i] = count
for i, count in zip(unique_bandit, counts_bandit):
    bandit_counts[i] = count

axes[0, 1].bar(x - width/2, policy_counts, width, label='Policy', alpha=0.8)
axes[0, 1].bar(x + width/2, bandit_counts, width, label='Bandit', alpha=0.8)
axes[0, 1].set_xlabel('Variant ID')
axes[0, 1].set_ylabel('Count')
axes[0, 1].set_title('Decision Distribution Comparison')
axes[0, 1].legend()

# Context diversity (first two principal components)
contexts_matrix = torch.stack(contexts).detach().cpu().numpy()
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
contexts_2d = pca.fit_transform(contexts_matrix)

scatter = axes[1, 0].scatter(contexts_2d[:, 0], contexts_2d[:, 1], 
                           c=policy_decisions, alpha=0.6, cmap='viridis')
axes[1, 0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
axes[1, 0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
axes[1, 0].set_title('Context Space Coloring by Policy Decision')
plt.colorbar(scatter, ax=axes[1, 0], label='Variant ID')

# Feature importance (simplified analysis)
feature_importance = np.random.rand(min(10, model.context_dim))  # Placeholder
axes[1, 1].bar(range(len(feature_importance)), feature_importance)
axes[1, 1].set_xlabel('Feature Index')
axes[1, 1].set_ylabel('Importance Score')
axes[1, 1].set_title('Feature Importance (Placeholder)')

plt.tight_layout()
plt.show()

## 5. Performance Analysis

In [ ]:
# Simulate performance under different scenarios
scenarios = {
    'low_load': {'batch_size': 1, 'priority': 1},
    'medium_load': {'batch_size': 4, 'priority': 3},
    'high_load': {'batch_size': 8, 'priority': 5},
    'high_accuracy': {'batch_size': 1, 'priority': 5, 'accuracy_requirement': 0.99}
}

scenario_results = {}

for scenario_name, request_params in scenarios.items():
    print(f"Simulating {scenario_name} scenario")
    
    metrics = RoutingMetrics(config._config)
    
    for _ in range(200):  # Fewer samples for interactive exploration
        context = torch.randn(model.context_dim)
        variant = model.select_model_variant(context, use_bandit=False)
        
        # Simulate performance
        performance = model.variant_manager.estimate_performance(variant, request_params)
        
        # Add scenario-specific noise
        if scenario_name == 'high_load':
            performance['latency_ms'] *= 1.3  # Higher latency under load
        elif scenario_name == 'high_accuracy':
            performance['accuracy'] *= 0.98  # Slight accuracy reduction
        
        # Calculate reward
        sla_constraints = config._config['routing']['sla_constraints']
        reward = model.variant_manager.calculate_reward(variant, performance, sla_constraints)
        
        metrics.record_request(
            latency_ms=performance['latency_ms'],
            accuracy=performance['accuracy'],
            memory_mb=performance['memory_mb'],
            throughput_rps=performance['throughput_rps'],
            variant_id=variant,
            reward=reward
        )
    
    scenario_results[scenario_name] = metrics.get_comprehensive_metrics()

print("Scenario analysis complete")

In [ ]:
# Visualize scenario performance
metrics_to_plot = ['p99_latency_ms', 'mean_accuracy', 'memory_reduction_pct', 'mean_reward']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, metric in enumerate(metrics_to_plot):
    scenarios_list = list(scenario_results.keys())
    values = [scenario_results[scenario].get(metric, 0) for scenario in scenarios_list]
    
    bars = axes[i].bar(scenarios_list, values, alpha=0.8)
    axes[i].set_title(metric.replace('_', ' ').title())
    axes[i].tick_params(axis='x', rotation=45)
    
    # Add value labels on bars
    for bar, value in zip(bars, values):
        height = bar.get_height()
        axes[i].text(bar.get_x() + bar.get_width()/2., height,
                    f'{value:.2f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

## 6. SLA Compliance Analysis

In [ ]:
# Analyze SLA compliance under different conditions
sla_constraints = config._config['routing']['sla_constraints']
print("SLA Constraints:")
for constraint, value in sla_constraints.items():
    print(f"  {constraint}: {value}")

# Test SLA compliance for each scenario
compliance_data = []

for scenario_name, results in scenario_results.items():
    latency_compliance = results.get('p99_latency_ms', 0) <= sla_constraints['p99_latency_ms']
    accuracy_compliance = results.get('mean_accuracy', 0) >= sla_constraints['accuracy_threshold']
    
    compliance_data.append({
        'scenario': scenario_name,
        'latency_ok': latency_compliance,
        'accuracy_ok': accuracy_compliance,
        'overall_ok': latency_compliance and accuracy_compliance
    })

# Create compliance visualization
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

scenarios_list = [d['scenario'] for d in compliance_data]
latency_ok = [d['latency_ok'] for d in compliance_data]
accuracy_ok = [d['accuracy_ok'] for d in compliance_data]
overall_ok = [d['overall_ok'] for d in compliance_data]

x = np.arange(len(scenarios_list))
width = 0.25

ax.bar(x - width, [int(ok) for ok in latency_ok], width, label='Latency SLA', alpha=0.8)
ax.bar(x, [int(ok) for ok in accuracy_ok], width, label='Accuracy SLA', alpha=0.8)
ax.bar(x + width, [int(ok) for ok in overall_ok], width, label='Overall SLA', alpha=0.8)

ax.set_xlabel('Scenario')
ax.set_ylabel('Compliance (1=Yes, 0=No)')
ax.set_title('SLA Compliance by Scenario')
ax.set_xticks(x)
ax.set_xticklabels(scenarios_list, rotation=45)
ax.legend()
ax.set_ylim(0, 1.1)

plt.tight_layout()
plt.show()

# Print summary
print("\nSLA Compliance Summary:")
for data in compliance_data:
    status = "✓" if data['overall_ok'] else "✗"
    print(f"{status} {data['scenario']}: Latency={data['latency_ok']}, Accuracy={data['accuracy_ok']}")

## 7. Interactive Model Exploration

In [ ]:
# Interactive function to test different contexts
def test_routing_decision(batch_size=1, image_size=224, priority=3, accuracy_req=0.95):
    """Test routing decision for given parameters."""
    
    # Create request context
    request_data = {
        'batch_size': batch_size,
        'image_size': image_size, 
        'priority': priority,
        'accuracy_requirement': accuracy_req
    }
    
    # Generate context vector (simplified)
    context = torch.randn(model.context_dim)
    
    # Get routing decisions
    policy_variant = model.select_model_variant(context, use_bandit=False)
    bandit_variant = model.select_model_variant(context, use_bandit=True)
    oracle_variant = model.variant_manager.get_best_variant_for_context(
        request_data, config._config['routing']['sla_constraints']
    )
    
    # Get performance estimates for each variant
    results = {}
    for name, variant in [('Policy', policy_variant), ('Bandit', bandit_variant), ('Oracle', oracle_variant)]:
        performance = model.variant_manager.estimate_performance(variant, request_data)
        variant_info = model.variant_manager.get_variant_info(variant)
        
        results[name] = {
            'variant': variant,
            'variant_name': variant_info['name'],
            'latency_ms': performance['latency_ms'],
            'accuracy': performance['accuracy'],
            'memory_mb': performance['memory_mb'],
            'throughput_rps': performance['throughput_rps']
        }
    
    # Print results
    print(f"Request: batch_size={batch_size}, image_size={image_size}, priority={priority}, accuracy_req={accuracy_req:.3f}")
    print("-" * 80)
    print(f"{'Method':<8} {'Variant':<8} {'Type':<8} {'Latency':<10} {'Accuracy':<10} {'Memory':<10} {'Throughput':<10}")
    print("-" * 80)
    
    for method, result in results.items():
        print(f"{method:<8} {result['variant']:<8} {result['variant_name']:<8} "
              f"{result['latency_ms']:<10.1f} {result['accuracy']:<10.3f} "
              f"{result['memory_mb']:<10.0f} {result['throughput_rps']:<10.1f}")
    
    return results

# Test different scenarios
print("Testing different request scenarios:\n")

# Low priority, small batch
print("Scenario 1: Low priority, small batch")
test_routing_decision(batch_size=1, priority=1)
print()

# High priority, large batch
print("Scenario 2: High priority, large batch")
test_routing_decision(batch_size=8, priority=5)
print()

# High accuracy requirement
print("Scenario 3: High accuracy requirement")
test_routing_decision(accuracy_req=0.99, priority=4)
print()

## 8. Summary and Insights

This notebook has demonstrated the key capabilities of the adaptive model routing system:

1. **Model Variants**: The system manages multiple compression variants with different memory/latency trade-offs
2. **Contextual Bandits**: Different exploration algorithms balance exploration vs exploitation
3. **Context-Aware Routing**: Routing decisions adapt based on request characteristics and system state
4. **Performance Analysis**: The system tracks multiple metrics and maintains SLA compliance
5. **Adaptive Learning**: The system learns from feedback to improve routing decisions over time

Key observations:
- Different exploration algorithms show varying convergence rates and final performance
- Policy and bandit decisions may disagree initially but should converge with training
- SLA compliance varies by scenario, demonstrating the need for adaptive routing
- The oracle provides an upper bound on achievable performance

For production deployment, consider:
- Fine-tuning exploration parameters based on your specific workload
- Implementing real-time monitoring and alerting
- Regular retraining as workload patterns change
- A/B testing between different routing strategies